# 51.0 FFT and Spectrogram of Original Audio

Load `data/audio/chirp_50_1000_3.0sec/audio.wav`, plot its FFT magnitude spectrum, then plot its spectrogram.

In [1]:
import sys
from pathlib import Path

def _find_repo_root(marker='pyproject.toml'):
    # repo layout/paths change across machines and over time -- walk up from cwd (usually
    # notebooks/) instead of hardcoding an absolute path.
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not find repo root (no {marker} found above {Path.cwd()})")

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
for p in (REPO_ROOT, SRC_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
from scipy.io import wavfile
from scipy.signal import spectrogram
import plotly.graph_objects as go

In [2]:
# ==== Parameters ====
AUDIO_PATH = REPO_ROOT / 'data' / 'audio' / 'chirp_50_1000_3.0sec' / 'audio.wav'

## 51.1 Load Audio

In [3]:
fs, signal = wavfile.read(AUDIO_PATH)

# Collapse to mono if needed
if signal.ndim > 1:
    signal = signal.mean(axis=1)

signal = signal.astype(np.float64)
n_samples = signal.shape[0]
duration = n_samples / fs
print(f"Sample rate: {fs} Hz")
print(f"Samples: {n_samples}")
print(f"Duration: {duration:.3f} sec")

Sample rate: 44100 Hz
Samples: 141120
Duration: 3.200 sec


## 51.2 Waveform

In [4]:
t = np.arange(n_samples) / fs

fig = go.Figure(go.Scatter(x=t, y=signal, mode='lines', line=dict(width=1)))
fig.update_layout(
    title='Waveform',
    xaxis_title='Time (s)',
    yaxis_title='Amplitude',
    width=900,
    height=350,
)
fig.show()

## 51.3 FFT

In [5]:
# Remove DC offset before taking the FFT
windowed = signal - signal.mean()

fft_vals = np.fft.rfft(windowed)
fft_freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
fft_mag = np.abs(fft_vals) / n_samples

In [6]:
fig = go.Figure(go.Scatter(x=fft_freqs, y=fft_mag, mode='lines', line=dict(width=1)))
fig.update_layout(
    title='FFT Magnitude Spectrum',
    xaxis_title='Frequency (Hz)',
    yaxis_title='Magnitude',
    xaxis=dict(range=[0, 5000]),
    width=900,
    height=400,
)
fig.show()

## 51.4 Spectrogram

In [7]:
freqs, times, Sxx = spectrogram(signal, fs=fs, nperseg=1024, noverlap=768)
Sxx_db = 10 * np.log10(Sxx + 1e-12)

In [8]:
fig = go.Figure(go.Heatmap(
    x=times,
    y=freqs,
    z=Sxx_db,
    colorscale='Viridis',
    colorbar=dict(title='dB'),
))
fig.update_layout(
    title='Spectrogram',
    xaxis_title='Time (s)',
    yaxis_title='Frequency (Hz)',
    yaxis=dict(range=[0, 5000]),
    width=900,
    height=450,
)
fig.show()